In [ ]:
# ── GPU / vLLM server setup ───────────────────────────────────────────────────
# Launch 4 vLLM servers (one per GPU) in a terminal on the GPU server
# before running this notebook. Copy-paste the block below:
#
# CUDA_VISIBLE_DEVICES=4 python3 -m vllm.entrypoints.openai.api_server --model meta-llama/Llama-3.1-8B-Instruct --port 8000 --gpu-memory-utilization 0.9 --max-model-len 8192 --dtype bfloat16 --api-key token-abc123 &
#
# CUDA_VISIBLE_DEVICES=5 python3 -m vllm.entrypoints.openai.api_server --model meta-llama/Llama-3.1-8B-Instruct --port 8001 --gpu-memory-utilization 0.9 --max-model-len 8192 --dtype bfloat16 --api-key token-abc123 &
#
# CUDA_VISIBLE_DEVICES=6 python3 -m vllm.entrypoints.openai.api_server --model meta-llama/Llama-3.1-8B-Instruct --port 8002 --gpu-memory-utilization 0.9 --max-model-len 8192 --dtype bfloat16 --api-key token-abc123 &
#
# CUDA_VISIBLE_DEVICES=7 python3 -m vllm.entrypoints.openai.api_server --model meta-llama/Llama-3.1-8B-Instruct --port 8003 --gpu-memory-utilization 0.9 --max-model-len 8192 --dtype bfloat16 --api-key token-abc123 &
#
# Wait ~60s for all 4 to finish loading, then run the cells below.
# To kill all servers: pkill -f "vllm.entrypoints.openai.api_server"

import os
from dotenv import load_dotenv
from pathlib import Path

MODEL = "llama"                         # this notebook's model folder (notebooks/llama/ → data/llama/)

load_dotenv(Path("../..").resolve() / ".env")

# ── Configure GPUs and ports here ────────────────────────────────────────────
#GPU_IDS    = [4, 5, 6, 7]
#VLLM_PORTS = [8000, 8001, 8002, 8003]

GPU_IDS    = [4, 5, 6]
VLLM_PORTS = [8000, 8001, 8002]

VLLM_BASE_URLS = [f"http://localhost:{p}/v1" for p in VLLM_PORTS]
print(f"GPUs:  {GPU_IDS}")
print(f"URLs:  {VLLM_BASE_URLS}")

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import json
import logging

import dotenv
import httpx
import pandas as pd
from datasets import load_dataset
from IPython.display import display

repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data_generation.generate_attacks import generate_all

dotenv.load_dotenv(repo_root / ".env")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s — %(message)s",
    datefmt="%H:%M:%S",
)
openai_key = os.environ.get("OPENAI_API_KEY", "")
print("Imports OK")

## 1. Environment checks

In [ ]:
# OpenAI key
assert openai_key, "OPENAI_API_KEY not set — fill in .env"
print(f"OpenAI key: {'*' * (len(openai_key) - 4)}{openai_key[-4:]}")

# Check all vLLM servers
import httpx
n_ready = 0
for url in VLLM_BASE_URLS:
    try:
        r = httpx.get(f"{url}/models", headers={"Authorization": "Bearer token-abc123"}, timeout=5)
        models = r.json()["data"]
        print(f"  {url}  OK — {[m['id'] for m in models]}")
        n_ready += 1
    except Exception as e:
        print(f"  {url}  NOT reachable: {e}")

print(f"\n{n_ready}/{len(VLLM_BASE_URLS)} vLLM servers ready")
if n_ready == 0:
    print("Launch servers with the commands in cell 1 before continuing.")
elif n_ready < len(VLLM_BASE_URLS):
    print("Some servers missing — generation will use only the available ones.")

## 2. Load WildJailbreak OOD selection

50 harmful + 50 benign goals selected for OOD distance from JBB
(see `notebooks/07_wildjailbreak_exploration.ipynb`). Loaded from
`data/selection/wildjailbreak_selected_v1.parquet`.

Pair IDs are reset to `0..49` within each split. Original WildJailbreak
indices and similarity scores are preserved as `wj_idx` and `max_cos_to_jbb`
on each objective for traceability.


In [ ]:
sel_path = repo_root / "data" / "selection" / "wildjailbreak_selected_v1.parquet"
sel = pd.read_parquet(sel_path)

def parse_objectives(df):
    df = df.sort_values("max_cos_to_jbb").reset_index(drop=True)
    out = []
    for i, row in df.iterrows():
        out.append({
            "objective":       row["vanilla"],
            "pair_id":         int(i),                       # 0..49 within the split
            "category":        "wildjailbreak",
            "behavior_id":     f"wj_{int(row['wj_idx']):05d}",
            "wj_idx":          int(row["wj_idx"]),
            "max_cos_to_jbb":  float(row["max_cos_to_jbb"]),
            "nearest_jbb_idx": int(row["nearest_jbb_idx"]),
        })
    return out

harmful_objectives = parse_objectives(sel[sel["goal_type"] == "harmful"])
benign_objectives  = parse_objectives(sel[sel["goal_type"] == "benign"])

df_h = pd.DataFrame(harmful_objectives)
df_b = pd.DataFrame(benign_objectives)

print(f"Harmful: {len(harmful_objectives)} objectives (sorted by max_cos asc — most OOD first)")
print(f"Benign:  {len(benign_objectives)} objectives")
print()
print("Harmful sample:")
display(df_h[["pair_id", "wj_idx", "max_cos_to_jbb", "objective"]].head(5))
print("Benign sample:")
display(df_b[["pair_id", "wj_idx", "max_cos_to_jbb", "objective"]].head(5))


## 3. Configuration

Set `framework` to the framework you want to run, then execute cells 4 and 5.
Run this notebook **three times** (once per framework) to generate all harmful + benign data.

- **Harmful:** 20 attempts × 50 objectives × max `n_turns` turns
- **Benign:** 20 attempts × 50 objectives, using the same framework as harmful

Set `max_objectives` to a small number (e.g. `3`) for a smoke test.

Output goes to `data/{model}/conversations_wj/` so JBB and WJ conversations
stay separated on disk.


In [ ]:
CONFIG = dict(
    framework        = "crescendo",   # ← change to "actorattack" or "xteaming" for other runs
    model_shortname  = "llama3_8b",
    target_model_id  = "meta-llama/Llama-3.1-8B-Instruct",

    n_turns            = 10,
    n_attempts_harmful = 20,
    n_attempts_benign  = 20,

    concurrency      = 50,   # per GPU — total in-flight = concurrency × n_gpus
    attacker_model   = os.environ.get("ATTACKER_MODEL", "gpt-4o"),
    judge_model      = os.environ.get("JUDGE_MODEL",    "gpt-4o"),

    resume           = True,

    # Set to a small number for a smoke test, None to run all 100
    max_objectives   = None,
)

fw = CONFIG["framework"]
CONFIG["harmful_output_dir"] = repo_root / "data" / MODEL / "conversations_wj" / f"{fw}_harmful"
CONFIG["benign_output_dir"]  = repo_root / "data" / MODEL / "conversations_wj" / f"{fw}_benign"

n_gpus = len(VLLM_BASE_URLS)
n_obj  = len(harmful_objectives) if CONFIG["max_objectives"] is None else CONFIG["max_objectives"]
print("Config:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nGPUs available: {n_gpus}")
print(f"Objectives per GPU: ~{n_obj // n_gpus} harmful + ~{n_obj // n_gpus} benign")
print(f"Total conversations: {n_obj * CONFIG['n_attempts_harmful']} harmful + "
      f"{n_obj * CONFIG['n_attempts_benign']} benign = "
      f"{n_obj * (CONFIG['n_attempts_harmful'] + CONFIG['n_attempts_benign'])}")

## 4. Generate harmful conversations

Each conversation is saved to `data/llama/conversations_wj/{framework}_harmful/`
as it completes — safe to interrupt and resume.


In [ ]:
import asyncio

subset_h = (
    harmful_objectives[:CONFIG["max_objectives"]]
    if CONFIG["max_objectives"] else harmful_objectives
)

# Split objectives across GPUs (round-robin keeps load balanced)
n_gpus  = len(VLLM_BASE_URLS)
chunks  = [subset_h[i::n_gpus] for i in range(n_gpus)]

print(f"Running {CONFIG['framework']} harmful — "
      f"{len(subset_h)} objectives × {CONFIG['n_attempts_harmful']} attempts "
      f"across {n_gpus} GPUs")
for i, (chunk, url) in enumerate(zip(chunks, VLLM_BASE_URLS)):
    print(f"  GPU {i} ({url}): {len(chunk)} objectives")

tasks = [
    generate_all(
        objectives      = chunk,
        framework       = CONFIG["framework"],
        model_shortname = CONFIG["model_shortname"],
        target_model_id = CONFIG["target_model_id"],
        goal_type       = "harmful",
        n_turns         = CONFIG["n_turns"],
        n_attempts      = CONFIG["n_attempts_harmful"],
        output_dir      = CONFIG["harmful_output_dir"],
        concurrency     = CONFIG["concurrency"],
        openai_api_key  = openai_key,
        vllm_base_url   = url,
        attacker_model  = CONFIG["attacker_model"],
        judge_model     = CONFIG["judge_model"],
        resume          = CONFIG["resume"],
        desc            = f"Harmful GPU{i}",
    )
    for i, (chunk, url) in enumerate(zip(chunks, VLLM_BASE_URLS))
]

results = await asyncio.gather(*tasks)
harmful_convs = [c for chunk_result in results for c in chunk_result]
print(f"\nDone — {len(harmful_convs)} harmful conversations returned.")

## 5. Generate benign conversations

Runs the same framework as harmful (from CONFIG) on the matched benign WildJailbreak objectives.
Output goes to `data/llama/conversations_wj/{framework}_benign/`.


In [ ]:
subset_b = (
    benign_objectives[:CONFIG["max_objectives"]]
    if CONFIG["max_objectives"] else benign_objectives
)

chunks_b = [subset_b[i::n_gpus] for i in range(n_gpus)]

print(f"Running {CONFIG['framework']} benign — "
      f"{len(subset_b)} objectives × {CONFIG['n_attempts_benign']} attempts "
      f"across {n_gpus} GPUs")

tasks = [
    generate_all(
        objectives      = chunk,
        framework       = CONFIG["framework"],
        model_shortname = CONFIG["model_shortname"],
        target_model_id = CONFIG["target_model_id"],
        goal_type       = "benign",
        n_turns         = CONFIG["n_turns"],
        n_attempts      = CONFIG["n_attempts_benign"],
        output_dir      = CONFIG["benign_output_dir"],
        concurrency     = CONFIG["concurrency"],
        openai_api_key  = openai_key,
        vllm_base_url   = url,
        attacker_model  = CONFIG["attacker_model"],
        judge_model     = CONFIG["judge_model"],
        resume          = CONFIG["resume"],
        desc            = f"Benign GPU{i}",
    )
    for i, (chunk, url) in enumerate(zip(chunks_b, VLLM_BASE_URLS))
]

results = await asyncio.gather(*tasks)
benign_convs = [c for chunk_result in results for c in chunk_result]
print(f"\nDone — {len(benign_convs)} benign conversations returned.")

## 6. Results summary

In [ ]:
def make_results_df(convs, objectives):
    obj_by_id = {o["pair_id"]: o for o in objectives}
    rows = []
    for c in convs:
        o = obj_by_id.get(c["objective_pair_id"], {})
        rows.append({
            "pair_id":        c["objective_pair_id"],
            "behavior_id":    o.get("behavior_id", ""),
            "category":       o.get("category", ""),
            "attempt":        c.get("attempt", 1),
            "verdict":        c["verdict"],
            "executed_turns": c["executed_turns"],
            "objective":      c["objective"][:80],
        })
    return pd.DataFrame(rows)

df_harm_res = make_results_df(harmful_convs, harmful_objectives)
df_ben_res  = make_results_df(benign_convs,  benign_objectives)

print("=== Harmful ===")
display(df_harm_res["verdict"].value_counts().rename("count").to_frame())
if len(df_harm_res):
    asr = (df_harm_res["verdict"] == "jailbroken").mean()
    # Per-objective ASR: objective is jailbroken if any attempt succeeded
    obj_asr = df_harm_res.groupby("pair_id")["verdict"].apply(lambda v: (v == "jailbroken").any()).mean()
    print(f"\nASR (attempt-level):    {asr:.1%}")
    print(f"ASR (objective-level):  {obj_asr:.1%}  (≥1 attempt jailbroken)")

print("\n=== Benign ===")
display(df_ben_res["verdict"].value_counts().rename("count").to_frame())

print("\n=== Turn depth distribution ===")
turns_summary = pd.concat([
    df_harm_res[["executed_turns"]].assign(split="harmful"),
    df_ben_res[["executed_turns"]].assign(split="benign"),
])
display(turns_summary.groupby("split")["executed_turns"].describe())

## 7. Inspect a conversation

In [ ]:
# Change these to inspect any conversation
SPLIT   = "harmful"   # "harmful" or "benign"
PAIR_ID = 0           # 0–99
ATTEMPT = 1           # 1–n_attempts (harmful only)

out_dir = CONFIG[f"{SPLIT}_output_dir"]
fname   = f"{CONFIG['framework']}_{CONFIG['model_shortname']}_pair{PAIR_ID:03d}_attempt{ATTEMPT}.json"
fpath   = out_dir / fname

if not fpath.exists():
    # Fallback: show any file matching pair_id
    matches = sorted(out_dir.glob(f"*_pair{PAIR_ID:03d}_*.json"))
    fpath = matches[0] if matches else None

if fpath and fpath.exists():
    conv = json.loads(fpath.read_text())
    print(f"File:      {fpath.name}")
    print(f"Objective: {conv['objective']}")
    print(f"Verdict:   {conv['verdict']} ({conv['executed_turns']}/{conv['n_turns']} turns)")
    print()
    for turn in conv["turns"]:
        role = turn["role"].upper()
        print(f"[Turn {turn['turn_idx']} — {role}]")
        print(turn["content"])
        print()
else:
    print(f"No file found for pair_id={PAIR_ID} in {out_dir}")